In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from src.db import DATA_RAW
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

df = pd.read_parquet(DATA_RAW / "cars.parquet")

# только числовые + log target
num_features = ["car_age", "mileage_km", "power_hp", "engine_volume_l"]
df_b = df.dropna(subset=num_features + ["price"]).copy()
print(f"baseline rows: {len(df_b)}")

X = df_b[num_features]
y = np.log1p(df_b["price"])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
pipe.fit(X_tr, y_tr)

pred_log = pipe.predict(X_te)
mae_log = mean_absolute_error(y_te, pred_log)
r2 = r2_score(y_te, pred_log)

# метрики в рублях
pred_rub = np.expm1(pred_log)
y_te_rub = np.expm1(y_te)
mae_rub = mean_absolute_error(y_te_rub, pred_rub)
mape = np.mean(np.abs((y_te_rub - pred_rub) / y_te_rub)) * 100

print(f"\nMAE (log)  : {mae_log:.4f}")
print(f"R²         : {r2:.4f}")
print(f"MAE (руб)  : {mae_rub:,.0f}")
print(f"MAPE       : {mape:.2f}%")

# коэффициенты
coefs = pd.DataFrame({
    "feature": num_features,
    "coef": pipe.named_steps["model"].coef_,
}).sort_values("coef", key=abs, ascending=False)
print("\nКоэффициенты:")
print(coefs.to_string(index=False))

baseline rows: 29633

MAE (log)  : 0.3373
R²         : 0.7631
MAE (руб)  : 1,123,966
MAPE       : 40.05%

Коэффициенты:
        feature      coef
       power_hp  0.414413
        car_age -0.397446
engine_volume_l  0.156035
     mileage_km -0.143929
